# 实验 3：用于图像的条件生成模型
> 中文阅读版。原始英文文件已保留，本文件主要将说明性文字翻译为中文，公式与代码结构保持不变。

欢迎来到实验 3！在前一个实验中，我们研究的是针对玩具二维数据分布的*无条件*生成。本次实验中，我们将研究基于 MNIST 手写数字数据集图像的*条件*生成。每张 MNIST 图像不再只是二维，而是 $32\times 32 = 1024$ 维！面对这个更具挑战性的设定，我们需要特别注意以下两点：
1. 为了处理*条件*生成，我们将使用*无分类器引导*（classifier-free guidance, CFG）（见第 2 部分 2.1）。
2. 为了给高维图像值数据上的已学习向量场建模，简单的 MLP 已经不够用了。相反，我们将采用*扩散 Transformer*（diffusion transformer）架构（见第 3 部分）。

如果你发现任何错误，或者有其他反馈，欢迎发邮件给我们：`ezraerives@gmail.com`、`phold@mit.edu`、`ronsh@mit.edu`。祝你学得开心！


In [1]:
import os
from abc import ABC, abstractmethod
from typing import Optional, List, Type, Tuple, Dict
import math
import uuid
import random

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.axes._axes import Axes
import torch
import torch.nn as nn
import torch.distributions as D
from torch.func import vmap, jacrev
from tqdm import tqdm
import seaborn as sns
from sklearn.datasets import make_moons, make_circles
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from einops import rearrange
from einops.layers.torch import Rearrange

### 第 0 部分：复用前几个实验中的组件
本节中，我们将重新导入实验一和实验二中的一些组件，并在此基础上做一些重要更新。首先，让我们回顾实验一和实验二中的 `Sampleable` 类。下面我们把它命名为 `OldSampleable`。


In [2]:
class Sampleable(ABC):
    """
    Distribution which can be sampled from
    """
    @abstractmethod
    def sample(self, num_samples: int) -> torch.Tensor:
        """
        Args:
            - num_samples: the desired number of samples
        Returns:
            - samples: b d
        """
        pass

正如我们很快会看到的，像 MNIST 这样的数据集既包含图像（这里是手写数字），也包含类别标签（取值为 0 到 9）。因此，我们会将 `Sampleable` 推广为 `LabeledSampleable`，以便同时容纳这些标签。旧的 `Sampleable.sample` 方法只返回 `samples: torch.Tensor`，而 `LabeledSampleable.sample` 将同时返回 `samples: torch.Tensor` *以及* `labels: Optional[torch.Tensor]`。这样一来，我们就在形式上把每个这样的 `Sampleable` 实例都看作是从“数据与标签的*联合分布*”中采样。下面我们实现新的 `LabeledSampleable`。


In [3]:
class LabeledSampleable(ABC):
    """
    Distribution which can be sampled from
    """
    @abstractmethod
    def sample(self, num_samples: int) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args:
            - num_samples: the desired number of samples
        Returns:
            - samples: b d
            - labels: b
        """
        pass

对于某些分布，例如高斯分布，用“标签”来描述它并不太合理。因此，我们会把高斯类实现为一个简单的 `Sampleable`。再次说明，把一个简单高斯分布“对象化”多少有些学院派意味，这样做主要是为了教学目的，同时也强调在我们的思维模型里，“分布”本身是一等对象。在实际工作中，更常见也更实用的做法通常是直接使用库函数，例如 `torch.randn` / `torch.randn_like`。


In [4]:
class IsotropicGaussian(nn.Module, Sampleable):
    """
    Sampleable wrapper around torch.randn
    """
    def __init__(self, shape: List[int], std: float = 1.0):
        """
        shape: shape of sampled data
        """
        super().__init__()
        self.shape = shape
        self.std = std
        self.dummy = nn.Buffer(torch.zeros(1)) # Will automatically be moved when self.to(...) is called...

    def sample(self, num_samples) -> torch.Tensor:
        return self.std * torch.randn(num_samples, *self.shape).to(self.dummy.device)

我们也会像前几个实验一样实现一个*高斯混合模型*（Gaussian mixture model, GMM），它的标签会自然地由所属的混合成分给出。在进入图像任务之前，GMM 能帮助我们先对条件训练和条件推断的实现做一个合理性检查（sanity check）。


In [ ]:
class GMM(nn.Module, LabeledSampleable):
  def __init__(self, means: torch.Tensor, covariances: torch.Tensor, weights: torch.Tensor):
    super().__init__()
    self.means = nn.Buffer(means)
    self.covariances = nn.Buffer(covariances)
    self.weights = nn.Buffer(weights)

  def sample(self, num_samples: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Args:
      - num_samples: the desired number of samples
    Returns:
      - samples: b n
      - labels: b
    """
    # Choose the amount of each mode
    # Perform multinomial sampling on CPU to avoid device-side assert errors
    labels = torch.multinomial(self.weights.cpu(), num_samples=num_samples, replacement=True).to(self.means.device)

    # Sample from each mode
    samples = torch.zeros(num_samples, self.means.shape[1]).to(self.means.device)
    for idx in range(len(self.means)):
      samples[labels == idx] = torch.randn_like(samples[labels == idx]) * self.covariances[idx] + self.means[idx]

    return samples, labels

接下来，在加入 `ConditionalProbabilityPath`（以及 `GaussianConditionalProbabilityPath`）时，我们会做两项更新：
1. 调整实现，使其能够处理 `LabeledSampleable` 中新增的标签。回忆一下，之前我们的条件变量记作 `z`，满足 $z \sim p_{\text{data}}(z)$。现在我们将同时采样 `z` 和标签 `y`，即 $(z,y) \sim p_{\text{data}}(z,y)$。
2. 确保逻辑能够兼容任意形状的数据，我们统一记作 `b ...`。视具体情形不同，它可能表示 `b d`（每个 batch 元素一个特征向量），也可能表示 `b c h w`（处理图像时的 batch、通道、高、宽）。


In [ ]:
class ConditionalProbabilityPath(nn.Module, ABC):
    """
    Abstract base class for conditional probability paths
    """
    def __init__(self, p_simple: Sampleable, p_data: LabeledSampleable):
        super().__init__()
        self.p_simple = p_simple
        self.p_data = p_data

    def sample_marginal_path(self, t: torch.Tensor) -> torch.Tensor:
        """
        Samples from the marginal distribution p_t(x) = p_t(x|z) p(z)
        Args:
            - t: b
        Returns:
            - x: samples from p_t(x), b ... (i.e.,. `b d`, `b c h w`, etc.)
        """
        num_samples = t.shape[0]
        # Sample conditioning variable z ~ p(z)
        z, _ = self.sample_conditioning_variable(num_samples) # (b ...)
        # Sample conditional probability path x ~ p_t(x|z)
        x = self.sample_conditional_path(z, t) # (b ...)
        return x

    @abstractmethod
    def sample_conditioning_variable(self, num_samples: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Samples the conditioning variable z and label y
        Args:
            - num_samples: the number of samples
        Returns:
            - z: b ...
            - y: b
        """
        pass

    @abstractmethod
    def sample_conditional_path(self, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Samples from the conditional distribution p_t(x|z)
        Args:
            - z: conditioning variable b ...
            - t: time b
        Returns:
            - x: samples from p_t(x|z), b ...
        """
        pass

    @abstractmethod
    def conditional_vector_field(self, x: torch.Tensor, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates the conditional vector field u_t(x|z)
        Args:
            - x: b ...
            - z: b ...
            - t: b
        Returns:
            - conditional_vector_field: conditional vector field b c h w
        """
        pass

    @abstractmethod
    def conditional_score(self, x: torch.Tensor, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates the conditional score of p_t(x|z)
        Args:
            - x: b ...
            - z: b ...
            - t: b
        Returns:
            - score: b ...
        """
        pass

最后，我们把 `GaussianConditionalProbabilityPath`、`LinearAlpha` 和 `LinearBeta` 也加回来，它们的定义与上一个实验基本类似。


In [ ]:
class Alpha(ABC):
    def __init__(self):
        # Check alpha_t(0) = 0
        assert torch.allclose(
            self(torch.zeros(1,)), torch.zeros(1,)
        )
        # Check alpha_1 = 1
        assert torch.allclose(
            self(torch.ones(1,)), torch.ones(1,)
        )

    @abstractmethod
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates alpha_t. Should satisfy: self(0.0) = 0.0, self(1.0) = 1.0.
        Args:
            - t: b
        Returns:
            - alpha_t: b
        """
        pass

    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates d/dt alpha_t.
        Args:
            - t: b
        Returns:
            - d/dt a_t: b
        """
        t = t.unsqueeze(1)
        dt = vmap(jacrev(self))(t)
        return dt.view(-1)

class Beta(ABC):
    def __init__(self):
        # Check beta_0 = 1
        assert torch.allclose(
            self(torch.zeros(1)), torch.ones(1)
        )
        # Check beta_1 = 0
        assert torch.allclose(
            self(torch.ones(1)), torch.zeros(1)
        )

    @abstractmethod
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates alpha_t. Should satisfy: self(0.0) = 1.0, self(1.0) = 0.0.
        Args:
            - t: b
        Returns:
            - beta_t: b
        """
        pass

    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates d/dt beta_t.
        Args:
            - t: b
        Returns:
            - d/dt beta_t: b
        """
        t = t.unsqueeze(1)
        dt = vmap(jacrev(self))(t)
        return dt.view(-1)

class LinearAlpha(Alpha):
    """
    Implements alpha_t = t
    """

    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - t: b
        Returns:
            - alpha_t: b
        """
        return t

    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates d/dt alpha_t.
        Args:
            - t: b
        Returns:
            - d/dt alpha_t b
        """
        return torch.ones_like(t)

class LinearBeta(Beta):
    """
    Implements beta_t = 1-t
    """
    def __call__(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            - t: b
        Returns:
            - beta_t: b
        """
        return 1-t

    def dt(self, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates d/dt alpha_t.
        Args:
            - t: b
        Returns:
            - d/dt alpha_t: b
        """
        return - torch.ones_like(t)

class GaussianConditionalProbabilityPath(ConditionalProbabilityPath):
    def __init__(self, p_data: Sampleable, p_simple_shape: List[int], alpha: Alpha, beta: Beta):
        p_simple = IsotropicGaussian(shape = p_simple_shape, std = 1.0)
        super().__init__(p_simple, p_data)
        self.alpha = alpha
        self.beta = beta
        self.rearrange_scalar = Rearrange(f'b -> b{" 1" * len(p_simple_shape)}')

    def sample_conditioning_variable(self, num_samples: int) -> torch.Tensor:
        """
        Samples the conditioning variable z and label y
        Args:
            - num_samples: the number of samples
        Returns:
            - z: b ...
            - y: b
        """
        return self.p_data.sample(num_samples)

    def sample_conditional_path(self, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Samples from the conditional distribution p_t(x|z)
        Args:
            - z: b ...
            - t: b
        Returns:
            - x: b ...
        """
        alpha_t = self.rearrange_scalar(self.alpha(t)) # (b 1 1 1)
        beta_t = self.rearrange_scalar(self.beta(t)) # (b 1 1 1)
        return alpha_t * z + beta_t * torch.randn_like(z)

    def conditional_vector_field(self, x: torch.Tensor, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates the conditional vector field u_t(x|z)
        Args:
            - x: b c h w
            - z: b c h w
            - t: b
        Returns:
            - conditional_vector_field: conditional vector field (num_samples, c, h, w)
        """
        alpha_t = self.rearrange_scalar(self.alpha(t)) # b
        beta_t = self.rearrange_scalar(self.beta(t)) # b
        dt_alpha_t = self.rearrange_scalar(self.alpha.dt(t)) # b
        dt_beta_t = self.rearrange_scalar(self.beta.dt(t)) # b

        return (dt_alpha_t - dt_beta_t / beta_t * alpha_t) * z + dt_beta_t / beta_t * x

    def conditional_score(self, x: torch.Tensor, z: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Evaluates the conditional score of p_t(x|z)
        Args:
            - x: b ...
            - z: b ...
            - t: b
        Returns:
            - conditional_score: b ...
        """
        alpha_t = self.rearrange_scalar(self.alpha(t))
        beta_t = self.rearrange_scalar(self.beta(t))
        return (z * alpha_t - x) / beta_t ** 2

现在，让我们相应地更新 `ODE`、`SDE` 和 `Simulator` 类。这基本上就是两件事：
1. 把 `xt: b d` 改成更一般的 `b ...`。
2. 为可选的*条件输入* `y: Optional[torch.Tensor]` 提供支持。为了更简洁，我们会在相关方法（如 `drift_coefficient`、`diffusion_coefficient`、`step`、`simulate` 等）的签名中加入通用的 `**kwargs`。


In [ ]:
class ODE(ABC):
    @abstractmethod
    def drift_coefficient(self, xt: torch.Tensor, t: torch.Tensor, **kwargs) -> torch.Tensor:
        """
        Returns the drift coefficient of the ODE.
        Args:
            - xt: b ...
            - t: b
        Returns:
            - drift_coefficient: b ...
        """
        pass

class SDE(ABC):
    @abstractmethod
    def drift_coefficient(self, xt: torch.Tensor, t: torch.Tensor, **kwargs) -> torch.Tensor:
        """
        Returns the drift coefficient of the ODE.
        Args:
            - xt: b ...
            - t: b
        Returns:
            - drift_coefficient: b ...
        """
        pass

    @abstractmethod
    def diffusion_coefficient(self, xt: torch.Tensor, t: torch.Tensor, **kwargs) -> torch.Tensor:
        """
        Returns the diffusion coefficient of the ODE.
        Args:
            - xt: b ...
            - t: b
        Returns:
            - diffusion_coefficient: b ...
        """
        pass

In [ ]:
class Simulator(ABC):
    @abstractmethod
    def step(self, xt: torch.Tensor, t: torch.Tensor, dt: torch.Tensor, **kwargs):
        """
        Takes one simulation step
        Args:
            - xt: b ...
            - t: b
            - dt: b
        Returns:
            - nxt: b ...
        """
        pass

    @torch.no_grad()
    def simulate(self, x: torch.Tensor, ts: torch.Tensor, use_tqdm: bool = True, **kwargs):
        """
        Simulates using the discretization gives by ts
        Args:
            - x_init: b ...
            - ts: b
        Returns:
            - x_final: b ...
        """
        nts = ts.shape[1]
        pbar = tqdm(range(nts - 1)) if use_tqdm else range(nts - 1)
        for t_idx in pbar:
            t = ts[:, t_idx]
            h = ts[:, t_idx + 1] - ts[:, t_idx]
            x = self.step(x, t, h, **kwargs)
        return x

    @torch.no_grad()
    def simulate_with_trajectory(self, x: torch.Tensor, ts: torch.Tensor, use_tqdm: bool = True, **kwargs):
        """
        Simulates using the discretization gives by ts
        Args:
            - x: b ...
            - ts: b nt
        Returns:
            - x_traj: b nt ...
        """
        x_traj = [x.clone()]
        nts = ts.shape[1]
        pbar = tqdm(range(nts - 1)) if use_tqdm else range(nts - 1)
        for t_idx in pbar:
            t = ts[:,t_idx]
            h = ts[:, t_idx + 1] - ts[:, t_idx]
            x = self.step(x, t, h, **kwargs)
            x_traj.append(x.clone())
        return torch.stack(x_traj, dim=1)

class EulerSimulator(Simulator):
    def __init__(self, ode: ODE):
        self.ode = ode

    def step(self, xt: torch.Tensor, t: torch.Tensor, h: torch.Tensor, **kwargs):
        h = h.view([-1] + [1] * (len(xt.shape) - 1))
        return xt + self.ode.drift_coefficient(xt, t, **kwargs) * h

class EulerMaruyamaSimulator(Simulator):
    def __init__(self, sde: SDE):
        self.sde = sde

    def step(self, xt: torch.Tensor, t: torch.Tensor, h: torch.Tensor, **kwargs):
        h = h.view([-1] + [1] * (len(xt.shape) - 1))
        return xt + self.sde.drift_coefficient(xt, t, **kwargs) * h + self.sde.diffusion_coefficient(xt, t, **kwargs) * torch.sqrt(h) * torch.randn_like(xt)

def record_every(num_timesteps: int, record_every: int) -> torch.Tensor:
    """
    Compute the indices to record in the trajectory given a record_every parameter
    """
    if record_every == 1:
        return torch.arange(num_timesteps)
    return torch.cat(
        [
            torch.arange(0, num_timesteps - 1, record_every),
            torch.tensor([num_timesteps - 1]),
        ]
    )

最后，把我们的 `Trainer` 定义也加回来。


In [ ]:
MiB = 1024 ** 2

def model_size_b(model: nn.Module) -> int:
    """
    Returns model size in bytes. Based on https://discuss.pytorch.org/t/finding-model-size/130275/2
    Args:
    - model: self-explanatory
    Returns:
    - size: model size in bytes
    """
    size = 0
    for param in model.parameters():
        size += param.nelement() * param.element_size()
    for buf in model.buffers():
        size += buf.nelement() * buf.element_size()
    return size


class Trainer(ABC):
    def __init__(
        self,
        **kwargs
      ):
        super().__init__()
        self.model = None
        self.opt = None
        self.output_dir = None

    @abstractmethod
    def get_train_loss(self, **kwargs) -> torch.Tensor:
        pass

    def checkpoint(self, step: int):
      pass

    def get_optimizer(self, lr: float):
        return torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=1e-4)

    def random_name(self) -> str:
        adjectives = ["autumn", "hidden", "bitter", "misty", "silent", "empty", "dry", "dark", "summer", "icy", "delicate", "quiet", "white", "cool", "spring", "winter", "patient"]
        foods = ["apple", "banana", "pear", "plum", "orange", "persimmon", "tangerine", "durian", "jackfruit", "jicama", "cantaloupe", "watermelon", "peach"]
        return f"{random.choice(adjectives)}-{random.choice(foods)}-{str(uuid.uuid4())[:8]}"

    def train(
        self,
        model: nn.Module,
        num_steps: int,
        lr: float = 1e-3,
        warmup_steps: int = 500,
        ckpt_every: Optional[int] = 500,
        run_name: Optional[str] = None,
        **kwargs
    ) -> Tuple[List[float], List[int]]:
        """
        Linear warmup from 0 -> lr over `warmup_steps`, then constant lr.
        """
        # Initialize run name and output directory
        run_name = run_name or self.random_name()
        self.output_dir = os.path.join("runs", run_name)
        os.makedirs(self.output_dir, exist_ok=False)
        print("Initialized output directory at: " + self.output_dir)

        # Grab size
        self.model = model
        size_b = model_size_b(self.model)
        print(f"Training model with size: {size_b / MiB:.3f} MiB")

        # Initialize optimizer and LR
        self.opt = self.get_optimizer(lr)
        self.model.train()

        for pg in self.opt.param_groups:
            pg["lr"] = 0.0

        # Main training loop
        losses: List[float] = []
        steps: List[int] = []

        pbar = tqdm(range(num_steps))
        for step in pbar:
            # Update LR
            if warmup_steps > 0 and step < warmup_steps:
                cur_lr = lr * float(step + 1) / float(warmup_steps)
            else:
                cur_lr = lr
            for pg in self.opt.param_groups:
                pg["lr"] = cur_lr

            # Forward + backward
            self.opt.zero_grad(set_to_none=True)
            loss = self.get_train_loss(**kwargs)
            loss.backward()

            # Take gradient step
            self.opt.step()

            losses.append(float(loss.detach().item()))
            steps.append(step)

            pbar.set_description(f"Step {step}, lr={cur_lr:.2e}, loss={loss.item():.4f}")

            # Callback if specified
            if ckpt_every is not None and step % ckpt_every == 0 and step > 0:
              self.model.eval()
              self.checkpoint(step)
              self.model.train()

        self.model.eval()
        return losses, list(range(num_steps))

# 第 1 部分：先熟悉一下 MNIST
在这一节中，我们先直观感受一下 MNIST。接着，我们会尝试使用 `ConditionalGaussianProbabilityPath` 为 MNIST 加噪。


In [ ]:
class MNISTSampler(nn.Module, LabeledSampleable):
    """
    Sampleable wrapper for the MNIST dataset
    """
    def __init__(self):
        super().__init__()
        self.dataset = datasets.MNIST(
            root='./data',
            train=True,
            download=True,
            transform=transforms.Compose([
                transforms.Resize((32, 32)),
                transforms.ToTensor(),
                transforms.Normalize((0.1305,), (0.2891,)),
            ])
        )
        self.dummy = nn.Buffer(torch.zeros(1)) # Will automatically be moved when self.to(...) is called...

    def sample(self, num_samples: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            - num_samples: the desired number of samples
        Returns:
            - samples: shape (batch_size, c, h, w)
            - labels: shape (batch_size, label_dim)
        """
        if num_samples > len(self.dataset):
            raise ValueError(f"num_samples exceeds dataset size: {len(self.dataset)}")

        indices = torch.randperm(len(self.dataset))[:num_samples]
        samples, labels = zip(*[self.dataset[i] for i in indices])
        samples = torch.stack(samples).to(self.dummy)
        labels = torch.tensor(labels, dtype=torch.int64).to(self.dummy.device)
        return samples, labels

现在来看看条件概率路径下的一些样本。


In [ ]:
# Change these!
num_rows = 3
num_cols = 3
num_timesteps = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize our sampler
sampler = MNISTSampler().to(device)

# Initialize probability path
path = GaussianConditionalProbabilityPath(
    p_data = MNISTSampler(),
    p_simple_shape = [1, 32, 32],
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)

# Sample
num_samples = num_rows * num_cols
z, _ = path.p_data.sample(num_samples)
z = z.view(-1, 1, 32, 32)

# Setup plot
fig, axes = plt.subplots(1, num_timesteps, figsize=(6 * num_cols * num_timesteps, 6 * num_rows))

# Sample from conditional probability paths and graph
ts = torch.linspace(0, 1, num_timesteps).to(device)
for tidx, t in enumerate(ts):
    tt = t.expand(num_samples) # b
    xt = path.sample_conditional_path(z, tt) # b 1 32 32
    grid = make_grid(xt, nrow=num_cols, normalize=True, value_range=(-1,1))
    axes[tidx].imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    axes[tidx].axis("off")
plt.show()

# 第 2 部分：无分类器引导


**引导（Guidance）**：在无条件生成中，我们只需要生成“任意一个”数字；而现在，我们希望能够指定，也就是*条件化*，我们想生成的是哪个数字。也就是说，我们希望模型能够理解“生成一张数字 8 的图像”，而不仅仅是“生成一张数字图像”。从这里开始，我们把希望生成的数字图像记作 $x \in \mathbb{R}^{1 \times 32 \times 32}$，把条件变量（这里是标签）记作 $y \in \{0, 1, \dots, 9\}$。如果我们先固定某个 $y$，并把数据分布看作 $p_{\text{simple}}(x|y)$，那么问题就退化成了一个无条件生成问题，此时我们可以例如通过条件流匹配目标来构建生成模型：
$$\begin{align*}\mathcal{L}_{\text{CFM}}^{\text{guided}}(\theta;y) &= \,\,\mathbb{E}_{\square} \lVert u_t^{\theta}(x|y) - u_t^{\text{ref}}(x|z)\rVert^2\\ \square &= z \sim p_{\text{data}}(z|y), x \sim p_t(x|z)\end{align*}$$
随后，我们只需要让 $y$ 也随数据一起变化（而不是固定它），并显式地让已学习的近似 `u_t^{\theta}(x|y)` 依赖于所选的 $y$。于是我们得到*带引导的*条件流匹配目标：
$$\begin{align*}\mathcal{L}_{\text{CFM}}(\theta) &= \,\,\mathbb{E}_{\square} \lVert u_t^{\theta}(x|y) - u_t^{\text{ref}}(x|z)\rVert^2\\ \square &= z,y \sim p_{\text{data}}(z,y), x \sim p_t(x|z)\end{align*}$$
注意，实践中 $(z,y) \sim p_{\text{simple}}(z,y)$ 就是指从带标签的数据集（这里是 MNIST）中采样图像 $z$ 和标签 $y$。到这里为止，一切都很自然；而且我们要强调的是，如果我们的目标只是从 $p_{\text{data}}(x|y)$ 中采样，那么理论上任务已经完成了。只不过在实际应用中，人们往往更关心图像的*感知质量*。为此，我们将推导一种称为*无分类器引导*（classifier-free guidance）的过程。


**无分类器引导（Classifier-Free Guidance）**：为了更直观，我们先在高斯概率路径的框架下推导引导，尽管最终结果完全可以合理地应用到任意概率路径上。回忆课堂内容：当 $(a_t, b_t) = \left(\frac{\dot{\alpha}_t}{\alpha_t}, -\frac{\dot{\beta}_t \beta_t \alpha_t - \dot{\alpha}_t \beta_t^2}{\alpha_t}\right)$ 时，有
$$u_t(x|y) = a_tx + b_t\nabla \log p_t(x|y).$$
这个恒等式把*条件边缘速度* $u_t(x|y)$ 与*条件 score* $\nabla \log p_t(x|y)$ 联系了起来。不过请注意，
$$\nabla \log p_t(x|y) = \nabla \log \left(\frac{p_t(x)p_t(y|x)}{p_t(y)}\right) = \nabla \log p_t(x) + \nabla \log p_t(y|x),$$
因此我们可以把它改写为
$$u_t(x|y) = a_tx + b_t(\nabla \log p_t(x) + \nabla \log p_t(y|x)) = u_t(x) + b_t \nabla \log p_t(y|x).$$
其中的 $\nabla \log p_t(y|x)$ 可以理解为某种“带噪声的分类器”项（这实际上正是*classifier guidance* 的来源，不过这里我们不讨论它）。经验上，人们发现当我们把这个分类器项的贡献放大时，条件效果往往更好，于是得到
$$\tilde{u}_t(x|y) = u_t(x) + w b_t \nabla \log p_t(y|x)$$
其中 $w > 1$ 被称为*引导强度*（guidance scale）。然后我们把
$b_t\log p_t(y|x) = u^{\text{target}}_t(x|y) - u^{\text{target}}_t(x)$
代入，就得到
$$\begin{align}\tilde{u}_t(x|y) &= u_t(x) + w b_t \nabla \log p_t(y|x)\\
&= u_t(x) + w (u^{\text{target}}_t(x|y) - u^{\text{target}}_t(x))\\
&= (1-w) u_t(x) + w u_t(x|y). \end{align}$$
因此，核心思想就是：同时训练 $u_t(x)$ 和条件模型 $u_t(x|y)$，然后在*推断阶段*把它们组合成 $\tilde{u}_t(x|y)$。于是我们的做法会是：
1. 使用条件流匹配训练 $u_t^{\theta} \approx u_t(x)$，以及条件模型 $u_t^{\theta}(x|y) \approx u_t(x|y)$。
2. 在推断时，使用 $\tilde{u}_t^{\theta}(x|y)$ 进行采样。

这时你可能会说：“等等，为什么一定要训练两个模型？” 的确，我们也可以把 $u_t(x)$ 看成是 $u_t(x|y)$ 在 $y=\varnothing$（表示*没有条件*）时的特殊情形。于是，我们只需在标签集合中额外加入一个新的空标签 $\varnothing$，让 $y \in \{0,1,\dots, 9, \varnothing\}$。这项技术就叫做 **classifier-free guidance**（CFG）。因此最终我们得到
$$\boxed{\tilde{u}_t(x|y) = (1-w) u_t(x|\varnothing) + w u_t(x|y)}.$$ 


**训练与 CFG**：现在我们必须修改条件流匹配目标，使其能够考虑 $y = \varnothing$ 的情况。当然，当我们从 MNIST 中采样 $(z,y)$ 时，永远不会真正得到 $y = \varnothing$，因此我们需要人为地把这种可能性加入进来。为此，我们定义一个超参数 $\eta$，表示我们丢弃原始标签 $y$ 并把它替换为 $\varnothing$ 的*概率*。实践中，例如可以直接设定 $\varnothing = 10$，因为它和 0 到 9 的数字标签足以区分开。在实现模型时，我们只需要能够把这个标签索引到某个嵌入里，例如通过 `torch.nn.Embedding`。于是我们得到 CFG 下的条件流匹配训练目标：
$$\begin{align*}\mathcal{L}_{\text{CFM}}(\theta) &= \,\,\mathbb{E}_{\square} \lVert u_t^{\theta}(x|y) - u_t^{\text{ref}}(x|z)\rVert^2\\
\square &= z,y \sim p_{\text{data}}(z,y), x \sim p_t(x|z),\,\text{replace $y$ with $\varnothing$ with probability $\eta$}\end{align*}$$
用普通语言来描述，这个目标就是：
1. 从 $p_{\text{data}}$（这里就是 MNIST）中采样一张图像 $z$ 和一个标签 $y$。
2. 以概率 $\eta$ 把标签 $y$ 替换为空标签 $\varnothing \triangleq 10$。
3. 从 $\mathcal{U}[0,1]$ 中采样 $t$。
4. 从条件概率路径 $p_t(x|z)$ 中采样 $x$。
5. 用 $u_t^{\theta}(x|y)$ 去回归 $u_t^{\text{ref}}(x|z)$。


### 问题 2.2：为无分类器引导构造训练过程
在这一节中，你将实现训练目标 $\mathcal{L}_{\text{CFM}}(\theta)$，其中 $u_t^{\theta}(x|y)$ 是下面所描述的 `ConditionalVectorField` 类的一个实例。


In [ ]:
class ConditionalVectorField(nn.Module, ABC):
    """
    Conditional vector field u_t^theta(x|y)
    """

    @abstractmethod
    def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor):
        """
        Args:
        - x: b ...
        - t: b
        - y: b
        Returns:
        - u_t^theta(x|y): b ...
        """
        pass

class CFGVectorFieldODE(ODE):
    def __init__(self, net: ConditionalVectorField, null_label: int, guidance_scale: float = 1.0):
        self.net = net
        self.guidance_scale = guidance_scale
        self.null_label = null_label

    def drift_coefficient(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """
        Args:
        - x: b ...
        - t: b
        - y: b
        """
        guided_vector_field = self.net(x, t, y)
        unguided_y = torch.ones_like(y) * self.null_label
        unguided_vector_field = self.net(x, t, unguided_y)
        return (1 - self.guidance_scale) * unguided_vector_field + self.guidance_scale * guided_vector_field


**你的任务**：补全 `CFGFlowTrainer.get_train_loss`，使其实现上面描述的 $\mathcal{L}_{\text{CFM}}(\theta)$。你可以直接把 $\varnothing = 10$ “写死”在代码里。更一般的实现本不应该做这个只针对 MNIST 的假设，但为了完成这份作业，你这样处理是可以的。我们建议把问题 2.2 和问题 2.3 一起完成，因为它们可以在 Sanity Check 2.4 中一起测试。

**提示**：
1. 要采样图像 $(z,y) \sim p_{\text{data}}$，使用 `self.path.p_data.sample`。
2. 可以通过 `mask = torch.rand(batch_size) < self.eta` 生成一个对应“概率 $\eta$”的掩码。
3. 可以用 `torch.rand(batch_size, 1, 1, 1)` 采样 $t \sim \mathcal{U}[0,1]`。不要把 `torch.rand` 和 `torch.randn` 搞混！
4. 可以用 `self.path.sample_conditional_path` 采样 $x \sim p_t(x|z)$。


In [ ]:
class CFGTrainer(Trainer):
    def __init__(self, path: GaussianConditionalProbabilityPath, eta: float, null_label: int, eps: float = 0.001, **kwargs):
        assert eta > 0 and eta < 1
        super().__init__(**kwargs)
        self.eta = eta
        self.eps = eps
        self.path = path
        self.null_label = null_label

    def get_train_loss(self, batch_size: int) -> torch.Tensor:
        # Step 1: Sample z,y from p_data
        raise NotImplementedError("Fill me in!")

        # Step 2: Set each label to 10 (i.e., null) with probability eta
        raise NotImplementedError("Fill me in!")

        # Step 3: Sample t and x
        raise NotImplementedError("Fill me in!")

        # Step 4: Regress and output loss
        raise NotImplementedError("Fill me in!")


为了对 `CFGTrainer` 的实现做一个合理性检查，我们先训练一个简单的、基于 MLP 的模型，让它能够从高斯混合分布中进行条件采样。首先，我们来实现 `MLPConditionalVectorField` 类。


### 问题 2.3：MLPConditionalVectorField
**你的任务：** 实现 `MLPConditionalVectorField.forward`。


In [ ]:
# Sanity check: implement MLPConditionalVectorField

class MLP(nn.Module):
  def __init__(self, dims: List[int], activation: Type[torch.nn.Module] = torch.nn.SiLU, final_init: bool = False):
    super().__init__()
    mlp = []
    for idx in range(len(dims) - 1):
        mlp.append(torch.nn.Linear(dims[idx], dims[idx + 1]))
        if idx < len(dims) - 2:
            mlp.append(activation())
    self.net = torch.nn.Sequential(*mlp)

    if final_init:
      nn.init.zeros_(self.net[-1].weight)
      nn.init.zeros_(self.net[-1].bias)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """
    Args:
    - x: b n d
    Returns:
    - x: b n d
    """
    return self.net(x)

class MLPConditionalVectorField(ConditionalVectorField):
  def __init__(
      self,
      dim: int,
      hidden_dim: int,
      class_dim: int,
      num_classes: int
    ):
    super().__init__()
    self.mlp = MLP([dim + class_dim + 1, hidden_dim, hidden_dim, dim])
    self.class_embedding = nn.Embedding(num_classes + 1, class_dim)

  def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor):
      """
      Args:
      - x: b d
      - t: b
      - y: b
      Returns:
      - u_t^theta(x|y): (b, c, h, w)
      """
      raise NotImplementedError("Fill me in!")


### 合理性检查 2.4
最后，让我们把问题 2.2 和 2.3 中的结果结合起来，用 `CFGTrainer`、`CFGVectorFieldODE` 和 `MLPConditionalVectorField` 从一个高斯混合模型（`GMM`）实例中进行采样，以验证实现是否合理。


In [ ]:
#######################################################################
# Train MLP-based Conditional Vector Field to target Gaussian mixture #
#######################################################################

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize GMM
angles = [0, 2 * math.pi / 3, 4 * math.pi / 3]
means = 2 * torch.tensor([[math.cos(a), math.sin(a)] for a in angles])
covs = torch.tensor([0.2, 0.2, 0.2])
weights = torch.tensor([1/3, 1/3, 1/3])
gmm = GMM(means, covs, weights).to(device)

# Initialize probability path
path = GaussianConditionalProbabilityPath(
    p_data = gmm,
    p_simple_shape = [2],
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)
vector_field = MLPConditionalVectorField(
    dim = 2,
    hidden_dim = 256,
    class_dim = 2,
    num_classes = 3
).to(device)

# Train vector field
trainer = CFGTrainer(
    path=path,
    eta=0.25,
    null_label=3,
)
losses, steps = trainer.train(model=vector_field, num_steps=3000, lr=1e-3, batch_size=250)
plt.plot(steps, losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.show()

In [ ]:
#####################
# Visualize Results #
#####################

# User knobs
guidance_strength = 1.0 # try changing me!

fig, axes = plt.subplots(1, 3, figsize=(6 * 3, 6))

# Panel 1: Target
ax = axes[0]
x_data, _ = gmm.sample(250)
x_data = x_data.detach().cpu().numpy()
ax.scatter(x_data[:, 0], x_data[:, 1], s=5, marker="*")
ax.set_title("Target")

# Panel 2: Conditioned on each mode
ax = axes[1]
cfg_vector_field = CFGVectorFieldODE(vector_field, guidance_scale=guidance_strength, null_label=3)
simulator = EulerSimulator(cfg_vector_field)

batch_size = 250
labels = torch.arange(3).repeat_interleave(batch_size).to(device)
x_init = path.p_simple.sample(3 * batch_size) # b 2
ts = torch.linspace(0, 1, 100).expand(3 * batch_size, -1).to(device) # b nt
xs = simulator.simulate(x_init, ts, y=labels) # b 2
for idx in range(3):
    xs_idx = xs[idx * batch_size: (idx + 1) * batch_size].detach().cpu().numpy()
    ax.scatter(xs_idx[:, 0], xs_idx[:, 1], s=5, label=f"Mode {idx}", marker="*")
ax.legend()
ax.set_title(f"CFG w/ Guidance Strength {guidance_strength:.2f}")

# Panel 3: Unconditioned
ax = axes[2]
batch_size = 750
labels = torch.ones(batch_size).long().to(device) * 3
x_init = path.p_simple.sample(batch_size) # b 2
ts = torch.linspace(0, 1, 100).expand(batch_size, -1).to(device) # b nt
xs = simulator.simulate(x_init, ts, y=labels).detach().cpu().numpy() # b 2
ax.scatter(xs[:, 0], xs[:, 1], s=5, label=f"Mode {idx}", marker="*")
ax.set_title(f"Unguided Samples")

# 第 3 部分：构建一个扩散 Transformer
到这里，我们已经讨论了无分类器引导，以及为了训练这类模型所必须考虑的关键点。现在剩下的问题是：具体应该选择什么模型。尤其是，前一个实验里对简单分布还够用的 MLP，在这里就不再合适了。为此，我们将逐个组件地实现一个 **diffusion transformer**！


### 问题 3.1：傅里叶时间编码器

首先，我们实现一个*傅里叶时间编码器*（Fourier time encoder），它把标量时间值 $t \in [0,1]$ 映射为
$$
    t^{\text{emb}} = \begin{bmatrix}
    \cos(2\pi w_1 t) & \cdots & \cos(2\pi w_d t) & \sin(2\pi w_1 t) & \cdots & \sin(2\pi w_d t)
    \end{bmatrix}^T,
$$
其中权重 $w_i \sim \mathcal{N}(0, 1)$ 来自单位高斯分布。

**你的任务**：实现 `FourierEncoder`。


In [ ]:
class FourierEncoder(nn.Module):
    """
    Based on https://github.com/lucidrains/denoising-diffusion-pytorch/blob/main/denoising_diffusion_pytorch/karras_unet.py#L183
    """
    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0
        self.half_dim = dim // 2
        self.weights = nn.Parameter(torch.randn(1, self.half_dim))

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
        - t: b
        Returns:
        - embeddings: b d
        """
        # Step 1: compute frequencies f_i = 2 * pi * w_i * t
        raise NotImplementedError("Fill me in!")

        # Step 2: compute sin(f_i) and cos(f_i)
        raise NotImplementedError("Fill me in!")

        # Step 3: Concatenate and return
        raise NotImplementedError("Fill me in!")


### 问题 3.2：Patchifier

`Patchifier` 接收一个形状为 `b c 32 32` 的图像张量，并将其*切分为 patch token*，得到形状 `b (h / p * w / p) d` 的表示，其中 `d` 表示 diffusion transformer 的隐藏维度（也就是下面实现里的 `dim`），`p` 表示 patch 大小，`h` 和 `w` 分别表示图像的高和宽。它分两步完成：
1. 先通过一个卷积层，把输入 `b c 32 32` 映射到 `b d h/p h/w`。
2. 再把 `b d h/p h/w` 重排为 `b (h/p h/w) d`（也就是一共 `n = h/p * h/w` 个 token，每个 token 的维度都是 `d`）。

注意，虽然在本节训练的标准像素空间 diffusion transformer 中 `c=1`，但到了第 5 节训练潜空间 diffusion transformer 时，`c` 就不再是 1 了。

**你的任务：** 实现 `Patchifier`。


In [ ]:
class Patchifier(nn.Module):
  def __init__(self, img_size: int, patch_size: int, c_in: int, dim: int):
    super().__init__()
    assert img_size % patch_size == 0, "Image size must be divisible by patch size"

    raise NotImplementedError("Fill me in!")


  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """
    Args:
    - x: (bs, 1, img_size, img_size)
    Returns:
    - x: (bs, 1, img_size, img_size)
    """
    raise NotImplementedError("Fill me in!")


![DiT 架构示意图](…18122 tokens truncated…HKTIj3kPWbFR9GfN1bjPlz5/fFs7ZNc+ObNu2zew04WNVelWCHUekmjVizqvTPlRcNmZ3vA+SebCfw/by/Hjte6vC5vTUqFEjcvToUY7e0XB/25x2PuYP9/ww3HeUl+7j95gbw/ubyelyGt/g2YOy28aNGyvbm+a4VfcGh6MSqMLKC6cvjHE0VRzyPcvjQzJXPuf7zcnIYtAZZ5zh5DRwe15ivHbt2so88P3hJPSoEjZ37lzbvlP8nDvdk+YwHn/8cVsaeN8/2ajqB+bN78oxY8bIzm3nYdQbtkhhkfIEIFClfBEiA24IqCo9pwF8N+Gp3ITVUPDaSAyjAnNqiPJ0X66Moxn+QlBubHTr1s3Ri9f8OwYU5UKqdbq9DEZ7ccuIVJ06vwUq3gBV7jC4WXOcB5Hle4cbQ7JR3Z+qQYwwGlKqtHAegnhWZA6JnvsxgJNIYzesd2zQZRT0/c7l7PU5D+Pe53QlUv5O5XLRRRdx0I7GaRPiWPs3dO/e3fZ+4dmbsgmjjuU4E2EnpxnnIAACIJAogaDayqrlybidJH+0pKoTVG07N/n0o33jJh4nN34LVMyLRQ/VqgLmNKjqFVmgYu7yclS87JqXWUWqQVlzWa1atco22MqDoHKZm9NuPuY2tLk/cOaZZ5ovK/syugNeUku1bxaHuWPHDt2Zb7+8KoQ8U4z5qEzZsmUt+ZL7Sry0s8rIs8pU7R0vz4/XvrcqbF6+LNr+Y3o+eFaYuSz5uHXr1vrlhH5VYfPSkW6M6h2xbt06m1d+LuTyZcF4yZIlNrcqC95vT86/SqAKIy96+sIYR1PFIQtULOCotn3g5S737t2rJ9fym0wClepjNi5rnvW0Z88eS7pjnfCyp/J9wqvOuDUsvsv+eUlLs1HVD+yHVzOJZcKoN2KlAddTkwAEqtQsN6TaIwFVpee3QBVWQ8FrIzGMCkzVEOUK7Msvv4xZUrz2tFxBqhrSekBe86/78/Kbap1uL4PRXtwyszAEqr/FpsPyPeDUWZPLkTd/5tlW+h8LnrJR3Z/mjjG7D6shpUpLUM+KzCHRc1XnjL8o9GISaeyG9Y4MuoyCvt+5PLw852Hd+5yuRMpfVS6jR4/mYKMalUDFS3jEMm4EqrDqWE5rIuxi5RXXQQAEQMArgaDayjwoKbcJ+VwWW1R1gty2c5tXP9o3buNSuVMJVPyxHs8ccfpr1aqVNnB//vnnR13KlmciLViwQBWtsl6RBSoVm88//1wZnpOlSjS54oorDOcqAYsHUN0aXq2DPzrR/3bv3m3xqurLsGgj9z+5zuf9JmX/lsB8OHnhhRds97g8k0cWOrjdwnsyy+2SP//805Ii2R/PuuJnSjZenh+vfW9V2O3bt5eT4HjOAqM5n04CjWMAPl7gJdOGDBminGWnEqh4H1Nz2vk41h5XcnJZBDCH4Vf+veZFT1cY42iqOFQCFadJtWcu3+e8ooFs3ApUPFuS332vv/66qz8vohI/t6pZmjyzjve+9Wr4mTffH3x8yy23eApm3rx5tjB69eplCUPV73ArtoZRb1gSi5MsQwACVZYpSmQkGgFVpReEQBUtDapr8TQUvDQSw6rAVA1RrpjcGi8NMS/5dxu/7C7VOt1eBqO9uGUuqk6d3wIVx8Nf1smNLT6vU6eO1lBMZA141f0pD2KE1ZBSpSWoZ4W5+mlUjPyYQeW2sRtPXuJ5x4ZRRkHe78zJy3Ouym8Q9z6nK5HyV6XTzbPDX0fLnf0GDRpwcqIa/sBC9vf0009b/IRVx3KkibCzJBonIAACIOAXgaDayv/884/t3cszXOTlj1R1gty2c5tNP9o3buNSuVMJVA0bNrQ0knhDWrdGNfgsdyRV96c8iBFWQ0qVFi+dNi/PiluGbt35MYATb2M3zHds0GUU9P3O5enlOQ/r3ud0xVv+bDfeclG9I15//XUOMqrh51Lv+Om/vMSL2YRVx3KcibAzpxnHIAACIOAHAb/ayjyophJU+L3bo0cPx6Sq6gS5befoWbrgR/tGCtLTqaq97GVw1hzZyy+/rNy/ddCgQWZn2rGbekXlhvfV2bZtmy08lcWcOXNsdWmVKlW05fh09/zBmbznLA8uqwQA3Y/++80339jCl+8btwIVh9moUSNbeDwb6ffff9ej9PWXxTS9jaH65YF3leEBaZV73W7AgAEqb4adl+fHa99bFbaXvk4QApVK2GOG/HGjm3uNZ1GZmfOxSqCqV6+ezd2WLVvMxRfzuFq1apYw5L5tWHnRExz0OBrHo4rDyzuQ3Z5//vlRnwkus6AEKtU7huMrU6ZMhGd4+W1Usz25P+fG8IeS8vuD37fybELVu5+3XXBjVH79rjfcpANuUo8ABKrUKzOkOA4CqkrPT4EqzIaC10ZiGBVYog1RL4PuXvMfx+0SSbVOt5fBaC9umR1vGKp3dvRf1VeYMmduwOjuzb8qgUpeDoE7Jm4bpcuXL7fF079/f0tyVPenPIgRVkNKlRYvnTYvz4oFgg8nfgzgJNLYNac5yHdsWGUU1P3OnLw852Hc+5yueMuf/cZbLkELVGHUsYmyY/8wIAACIOAnAVVb2dwei/e4QIECkffffz9mUlV1gqptFzMg4cCP9o2beJzcqASqePnJ/lq1aqUtu62K20udzPtOyWHzOQ+IsuDIH99w24Pb+RynU574ow4n07VrV2UcHBav3sADtixw8l4vRYsWtbllUYA/aJKNV4GKBTSVgFq4cOHIqlWr5OATOj9y5IitL6Nzbt68uWPYW7ZsUYog7JefzVjGy/PDHw3qadJ/edm6c845R9sji8dUzEYVtpe+ThACFadPNVuT75kbbrghMm7cuMjMmTMj4z8Yr4kc8hJoer7Nv7w0KYtYqlluqn2Y2S/fy/x88H3M+02xoKISGszxyAJV2HlRjaOZ0xfPsSyiquKIR6DiWWqqWZV6Gv0WqGbNmmV7NvS4/P6VV3/ij4hVHw7wu5E/FuH90HiVCn4387tVtXwkp7Fjx458S9mMl/rB5vl/FmHUG05xwz41CUCgSu3yQ+pdElBVen4KVGE2FLw0EsOqwBJtiHoZdPeSf5e3h81ZqnW6vQxGe3HLYFSdOredDHmJCm6geRWo5MLh/aaGDBmiLbkgL6OgNwD/lpb6U92fcqctrIaUKi1ueTILL8+KzC7Rcz8GcOJt7Ib5jk2mMornfudy9vKch3Xvc7riLX/2G2+5BClQhVXHJsqO/cOAAAiAgJ8Egmgr33///bZ2HosXKqOqE+S2ncqfys6P9o0qXLd2fgtUvMSV3D7mJeRk46ZO5hUK5LD0pejk8JzO586dawvjvvvuM5zzjAc5jmXLlhnXYx2waHP8+HHjTx7YVvVleECX/elGXnKP08OzYX766Sfdie+/vHSlOd9169bV4pCXQ2QBzWx4/xqzP96HioU2eXm5aM+Dl+fHa99bFbaXvk4QApVKwOCZd26Mak9l5qy6P1VtR957aM2aNTGj4mUqVcvZywJVWHnRExr0OBrHo4rDrUDF/X5V3dGzZ89QBSoW8rNrhiqH7eV+YPcqE0a94RQ37FOXAAQq1yo/pD4lAVWl52fFGlajx2sjkVEFXYEl2hD1MugeT/493i7KJf4SvV/Yf1Cdbi+D0V7c6ty446HKPzeAeBNebqB36NBB+aWk7E8lUHE83AmT3fI5L+/B+2Bxp4GXUuDNsidPnhxp1qyZcho/L3khG9X96TSIEXRDSpUWL502L8+KzCHRcz8GcBJp7Ib1jg2jjIK837mc43nOg773OV2JlH+85RK0QMX5CrqOTZQd+4cBARAAAT8JBCFQ8Swgt0siqeoEp7ZdrHz70b6JFUe0605ijqpd7MXu1ltvjbosnZc6mb/ed5op4DZN3F9QzZ7S2fzzzz9KYcht+E57CnkVqDg93O9QzYLhfo/fhvs6qjy+9NJLUaNSiWgczieffBLVH1/08vxw30vFQk8zz1ozG1XYXvo6QQlUvBQbC2t6ur38sojF+/So/Fx88cXm7GvH3GflFUdU7mPZyQKLSqAKMy9hjKOp4ohHoGL4qna/zjwrCVSc127dusV1j+k8+INj1QxVDttL/cDuVSaMekMVL+xSlwAEqtQtO6TcAwFVpae/mOP9lQWqsBoKXhuJOqYgK7BEG6JeBt3jzb/Owc1vqnW6vQxGe3Grs+IGimrZjVjPDjfY5U6Gk0DFM2Tk2VCxwpevn3nmmcr9ClT3p9MgRtANKVVavHTavDwrevn59evHAE4ijd2w3rFhlFGQ9zuXdzzPedD3PqcrkfKPt1xUHVVeGjSW4edSfsfwDF4nE2Qdy3Emws4pzbAHARAAgXgJ+NlW5gHuXr16aR8iuU2Pqk5watvFCtOP9k2sOKJd91ugqlatmnImkZwGr/UKzx6SB8/letLpnMUpeek9OT18zkuIq5YGdAqX7XkWAc+qchrQjkeg4rQ4LYfFMxL8NLzcmSp/sfaRUrU7eNWLWHsdcdq9Pj+NGjVSppHTnSoCFef7kUceccyHqgy4//vQQw9psxDr1aun9HvhhRdy0DbD+yRHW1VEFd/tt98eYWHSfE0lUHFkYeUljHE0VRxOz7MNtMKCZxCZGerHWU2g4qz37dtXmVc9z06/LLZzf9TJeK0fnMIJo95wihv2qUcAAlXqlRlSHAcBVaXn9LJ2ay8LVJyssBoKXhqJZlxBVWCqRm6Qg+7x5t/MItpxqnW6vQxGe3FrZjRkyJCoX8/Jzw3PduFOprx2t5NAxXGNGTNGuZygHLbqnDv38+bNMyfZOFbdn9EGMYJsSKnSEuSzYkDw4cCPAZxEG7thvGPDKqOg7ncu6nif8yDvfU5XIuUfb7mEJVBx/oKqYxNlx/5hQAAEQMBPAtH2TFG103Q7HkDnQUIeHONZ7zxL3s2Aupz2Tz/91DYoN2PGDNmZq3NV+4YFs7AM89D5ePllYYbFAf5A67LLLtMG0vljHrcDuwsXLrTFy0tCRTO8txSz4VUh3KSVB+h5GTQvhmdZcZkUL148ZhwNGzaMLFiwIGrwvAeXOa0crhuzY8cO24d2HI5qDMBNeE5uNmzYYOtj8Wy1WIb3jTHni495XzA3xuvzw3vt8DMrx8fnskClCpuXknZr5I8bnQQat+HJ7phbrCX8eKYVs+RnRDe8RHf58uVtDJwEKvbH9xDPZFRxM9vxPak/e7KgGi3/YeRFtdebOe3xHPOyhmYjx8EfrfIS2vEap2f3/PPPjzdIpb/ffvstZtnGw0flhwVPJ8NjCW6FfX4nDxgwICbfeOoHp/SFUW84xQ371CIAgSq1ygupjZOAXOmpXvpe7eSKVU9aGA0FL41EPV36bxAVWKINUa+zQhLJv84h2m+qdbp5ORT5/pXXKdbz68Wt7kf/5YHeWJ1oXp6FNxHWv5I86//bu2OcOLIgDMAFZ3ECh0AkSJBxEZBISDkCGQfgBqTriIzUS0CIuAAZMa5qCYldY+yiBhavvpHQoGFq+vU33UxP/6/f+/LlH217LaCq5VxfX/9yLO7n61onOuqk8L/nnXpqc92/tH3+6iTGex1IvdSWzpe27r7y3GH6+ypO4KziYPe9/8d+5Hv0Htt7vc+T/fy9tv1q1+T9f+v7ssqA6vT0tFbj1dt7fMbWAid2rzbYHwkQIECAQFOgOqGdnZ0t8+nUCfQ63q+rq6pTaF0tVXPr1FyxvxuWvbT4mp+phqs7Ojpawsx63bqq6eTk5LE6mtUxlNvHCtze3j6en58v89rUXMEXFxePNR/Wn3arOZYrEKqwtYKoGnp7e3t7uRKv5of62ZUlFXzUPFXHx8dLx+QKk35nO6xwoTrZ1bI2Nzcf6yrHCvwqvKoOa28J6Z/MP3pdnpbr/nMJVKBXHRQODw8fK7ivwKo6DFfnhZr37ODgYBn+cBL8Tdf4Iz43pm1U/98KrNXi82SfGwECKxTIk+WRvb0jD1ji5uYm8vLwqMey103kWK+RkxFGXvHxwxJz7O34+tfX+Pb3t1hbW4u8+iR2dnYiJyH94bn1QI5vHBlELK+d4yJH9mCKHFYtstfVi89/erB2+5zUMXIi2cjhA6KWm4FCZLiw1GbPitjd3Y29vb2lHU91n+3+rev/2dbjT2vP1dVVZMATeZARGRAt208eAMXGxkbkwX1kr6fRKtX2maFN5En2uLy8jLu7u3h4eIjsUbls4xnSRAZly09OQhvZk3G0vNeKMwxd9pNqS35Zifv7+8gv28s61v5W67y/vx9bW1uxvr7+2kv52woFPup/7Aqb/NOX+kzb+/NG2vafa/R+/798xvbW2rMJECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgMBAdUATykBAgQIECBAgAABAgQIECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgMBAdUATykBAgQIECBAgAABAgQIECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgMBAdUATykBAgQIECBAgAABAgQIECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgMBAdUATykBAgQIECBAgAABAgQIECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgMBAdUATykBAgQIECBAgAABAgQIECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgMBAdUATykBAgQIECBAgAABAgQIECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgMBAdUATykBAgQIECBAgAABAgQIECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgMBAdUATykBAgQIECBAgAABAgQIECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgMBAdUATykBAgQIECBAgAABAgQIECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgMBAdUATykBAgQIECBAgAABAgQIECBAgAABAgQIECBAgEBfQEDVN1NBgAABAgQIECBAgAABAgQIECBAgAABAgQIECAwEBBQDfCUEiBAgAABAgQIECBAgAABAgQIECBAgAABAgQI9AUEVH0zFQQIECBAgAABAgQIECBAgAABAgQIECBAgAABAgOB7zXOOGxVxQslAAAAAElFTkSuQmCC)


### 问题 3.3：扩散 Transformer
现在，我们的数据已经变成了 `b n d` 的形状，其中 `n` 表示每张图像对应的 token 数量。接下来我们将把它送入一个 transformer。抱着“撞墙式学习”的精神，这一整套你都将亲手实现！特别是，如果你之前还没有从零写过 attention，那么这会是一次非常有价值的练习。完成这道题时，你可以参考上面的 diffusion transformer 架构图（摘自参考文献 [1]）。

**你的任务**：完成 `MHA`（多头自注意力）、`DiffusionTransformerLayer` 和 `DiffusionTransformer`。

**推荐步骤**：我们建议从上往下拆着做。
1. 先实现 `DiffusionTransformer`。尤其推荐使用固定的、每个位置一个的可学习位置编码，例如 `nn.Parameter(torch.randn(n_tokens, dim))`。注意，你可以通过参数 `n_tokens` 获得 token 的固定数量。此外，你还需要初始化 `depth` 个 `DiffusionTransformerLayer` 实例。前向传播时，只需要把位置编码加到输入上（注意广播维度！），再依次通过各层 diffusion transformer 即可。
2. 然后实现 `DiffusionTransformerLayer`。这里请参考上面的结构图。注意，adaLN-Zero 指的是把条件 MLP 的权重初始化为零（最后一层初始化为零就足够了）。这一步虽然不是必须的，但把这些残差连接初始化为零通常有助于稳定训练。还要注意，$\gamma, \beta, \alpha$ 的形状应为 `b d`（小心广播），而 scale-and-shift 通常写成这样的调制形式：$x \mapsto x * (1 + \gamma) + \beta$。前馈网络部分可以直接复用前面定义过的 `MLP` 类。DiT 中常见的隐藏维度配置是 `[dim, 4 * dim, dim]`。
3. 最后，实现多头自注意力。关于这一步，互联网会是你的朋友 :)

我们非常建议你**不要**使用 ChatGPT、Gemini、Claude 或其他大语言模型来替你写这段代码。这份实验本来就是选做的，如果把这部分也交给模型来完成，你其实失去的是一次被认真设计过的、亲手做机器学习实现的机会。


In [ ]:
class MHA(nn.Module):
  """
  Multi-headed self-attention
  """
  def __init__(self, dim: int, heads: int):
    super().__init__()
    assert dim % heads == 0

    raise NotImplementedError("Fill me in!")


  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """
    Args:
    - x: b n d
    Returns:
    - x: b n d
    """
    # Compute queries, keys, and values
    raise NotImplementedError("Fill me in!")

    # Fold head into batch dimension
    raise NotImplementedError("Fill me in!")

    # Compute attention
    raise NotImplementedError("Fill me in!")


    # Combine with values
    raise NotImplementedError("Fill me in!")

    # Unfold heads
    raise NotImplementedError("Fill me in!")

    # Pass throuh FF and return
    raise NotImplementedError("Fill me in!")

class DiffusionTransformerLayer(nn.Module):
  def __init__(
      self,
      dim: int,
      heads: int,
  ):
    """
    Args:
    - n_tokens: sequence length (for sake of positional embeddings)
    - dim: dimension of hidden layers
    - heads: number of attention heads
    """
    super().__init__()

    # Init normalization
    raise NotImplementedError("Fill me in!")


    # Initialize conditioning to zero - stabilizes residual connection!
    raise NotImplementedError("Fill me in!")


    # Init attention
    raise NotImplementedError("Fill me in!")

    # Init feedforward
    raise NotImplementedError("Fill me in!")

  def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """
    Args:
    - x: b n d
    - c: b d
    Returns:
    - x: b n d
    """
    # Compute conditioning gating, scaling, and bias
    raise NotImplementedError("Fill me in!")

    # Attention + residual connection
    raise NotImplementedError("Fill me in!")

    # Feedforward + residual connection
    raise NotImplementedError("Fill me in!")


class DiffusionTransformer(nn.Module):
  def __init__(
      self,
      depth: int,
      n_tokens: int,
      dim: int,
      **layer_kwargs,
  ):
    """
    Args:
    - n_tokens: sequence length (for sake of positional embeddings)
    - dim: dimension of hidden layers
    - heads: number of attention heads
    - depth: number of layers
    """
    super().__init__()
    raise NotImplementedError("Fill me in!")


  def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """
    Args:
    - x: b n d
    - c: b d
    Returns:
    - x: b n d
    """
    raise NotImplementedError("Fill me in!")


### 问题 3.4：Depatchifier
在 diffusion transformer 之后，我们需要把张量从 `b n d` 转回 `b 1 h w`。

**你的任务：** 实现 `Depatchifier`。

**提示**：一种可行的方法是：
1. 先做某种归一化，`nn.LayerNorm` 或 `nn.RMSNorm` 都可以。
2. 再通过某个 MLP，把张量映射到 `b (h/p w/p) (f p p)`（也就是说，把维度映射到 $d=fp^2$，其中 patch 大小为 $p$，“最终维度”为 $f$）。
3. 将其重排为 `b f h w`。
4. 最后再通过一个卷积层，得到形状为 `b 1 h w` 的输出。


In [ ]:
class Depatchifier(nn.Module):
  def __init__(self, img_size: int, patch_size: int, dim: int, final_dim: int, c_out: int):
    super().__init__()
    self.patch_size = patch_size
    assert img_size % patch_size == 0, "Image size must be divisible by patch size"

    raise NotImplementedError("Fill me in!")


  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """
    Args:
    - x: b n d
    Returns:
    - x: b 1 32 32
    """
    raise NotImplementedError("Fill me in!")


### 问题 3.5：把所有组件拼起来
最后，我们来实现基于 DiT 的引导流模型 $u_t^\theta(x|y)$，对应类 `DiffusionTransformerFlowModel`。

**你的任务**：实现 `DiffusionTransformerFlowModel`。

**提示**：
1. 要嵌入引导输入 `y`，你需要对类别标签做 embedding。这里可以复用你在 `MLPConditionalVectorField` 中的思路。尤其推荐使用 `nn.Embedding(num_classes=11, embedding_dim=dim)`。这里是 11 个类而不是 10 个，因为我们还要额外考虑空标签。
2. 按照 DiT 总览图来做：先分别嵌入 $t$ 和 $y$，然后把它们相加得到引导变量。接着用 `patchifier` 处理 $x$，再把 patchifier 的输出和 $t + y$ 一起送入 diffusion transformer。最后，再把 DiT 的输出交给 depatchifier。


In [ ]:
class DiffusionTransformerFlowModel(ConditionalVectorField):
  def __init__(
      self,
      img_size: int = 32,
      patch_size: int = 8,
      num_layers: int = 12,
      c: int = 1,
      dim: int = 256,
      heads: int = 4,
      final_dim: int = 10,
      n_classes: int = 11,
    ):
      super().__init__()
      # 0. Construct time_embedder and y_embedder
      self.time_embedder = None # Todo!
      self.y_embedder = None # Todo!

      # 1. Construct patchifier
      self.patchifier = None # Todo!

      # 2. Construct DiT
      n_tokens = (img_size // patch_size) ** 2
      self.dit = None # Todo!

      # 3. Construct de-patchifier
      self.depatchifier = None # Todo!

  def forward(self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Args:
    - x: b 1 32 32
    - t: b 1 1 1
    - c: b 1 1 1
    Returns:
    - u_t^theta(x|y): b 1 32 32
    """
    # 1. Embed time and y
    raise NotImplementedError("Fill me in!")


    # 2. Patchify
    raise NotImplementedError("Fill me in!")

    # 3. Pass through DiT
    raise NotImplementedError("Fill me in!")

    # 4. Depatchify
    raise NotImplementedError("Fill me in!")

    return x

### 训练扩散 Transformer


In [ ]:
##################
# Training utils #
##################

@torch.no_grad()
def visualize_output(model, path, samples_per_class: int = 10, num_timesteps: int = 100, guidance_scales: List[float] = [1.0, 3.0, 5.0], save_path: Optional[str] = None, use_tqdm: bool = True):
  # Graph
  fig, axes = plt.subplots(1, len(guidance_scales), figsize=(10 * len(guidance_scales), 10))

  for idx, w in enumerate(guidance_scales):
      # Setup ode and simulator
      ode = CFGVectorFieldODE(model, guidance_scale=w, null_label=10)
      simulator = EulerSimulator(ode)

      # Sample initial conditions
      y = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=torch.int64).repeat_interleave(samples_per_class).to(device)
      num_samples = y.shape[0]
      x0 = path.p_simple.sample(num_samples) # (num_samples, 1, 32, 32)

      # Simulate
      ts = torch.linspace(0,0.999,num_timesteps).view(1, -1, 1, 1, 1).expand(num_samples, -1, 1, 1, 1).to(device)
      x1 = simulator.simulate(x0, ts, y=y, use_tqdm=use_tqdm)

      # Plot
      v_min, v_max = x1.min(), x1.max()
      x1 = (x1 - v_min) / (v_max - v_min)
      grid = make_grid(x1, nrow=samples_per_class, normalize=True, value_range=(0,1))
      axes[idx].imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
      axes[idx].axis("off")
      axes[idx].set_title(f"Guidance: $w={w:.1f}$", fontsize=25)

  # Save
  if save_path is not None:
      plt.savefig(save_path)
      plt.close()
  else:
    plt.show()

class MNISTCFGTrainer(CFGTrainer):
  '''
  CFG Trainer with MNIST-specific callback
  '''
  def checkpoint(self, step: int):
    # Save model
    torch.save(self.model.state_dict(), os.path.join(self.output_dir, f'step_{step:6d}_model.pt'))
    torch.save(self.opt.state_dict(), os.path.join(self.output_dir, f'step_{step:6d}_opt.pt'))

    # Save output visualization
    visualize_output(self.model, self.path, save_path=os.path.join(self.output_dir, f'step_{step:6d}_output.png'), use_tqdm=False)

现在终于可以训练我们的模型了！注意，你可以通过查看指定运行目录中的中间输出和可视化结果来跟踪训练进度。


In [ ]:
#################
# Training code #
#################

# Initialize probability path
path = GaussianConditionalProbabilityPath(
    p_data = MNISTSampler(),
    p_simple_shape = [1, 32, 32],
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)

# Initialize model
dit = DiffusionTransformerFlowModel(
    img_size = 32,
    patch_size = 4,
    num_layers = 8,
    c = 1,
    dim = 256,
    heads = 8,
    final_dim = 10,
    n_classes = 11,
).to(device)

# Initialize trainer
trainer = MNISTCFGTrainer(path = path, eta=0.35, null_label=10)

# Train! You should have reasonable results in ~15 A100 minutes
losses, steps = trainer.train(model=dit, num_steps = 20000, lr=0.4e-3, batch_size=256, ckpt_every=1000)

plt.plot(steps, losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Loss vs. Step")
plt.show()

现在来看看最终模型的输出！


In [ ]:
# Play with these!
samples_per_class = 10
num_timesteps = 100
guidance_scales = [1.0, 3.0, 5.0]

visualize_output(
    model=dit,
    path=path,
    samples_per_class=samples_per_class,
    num_timesteps=num_timesteps,
    guidance_scales=guidance_scales,
)
plt.show()

# 第 4 部分：训练一个变分自编码器（VAE）
在这一节中，我们将为 MNIST 训练一个变分自编码器（VAE）。下一节里，我们会在学到的潜空间中训练一个 diffusion transformer。回忆讲义中的内容，VAE 的整体结构由以下几部分组成：
1. 一个*编码器* $q_\phi(z|x)$，它把形状为 `b 1 32 32` 的输入 `x` 映射为 `z_mean`（形状为 `b c h w`）以及一个可学习的标量 `z_logvar`。注意，出于训练稳定性的考虑，并且沿用参考文献 [2] 的做法，我们选择间接地参数化对数方差 $\log \sigma_\phi(x)$。
2. 一个*解码器* $p_\theta(x|z)$，它以类似方式把潜变量 `z` 映射为 `x_mean`（形状为 `b 1 32 32`）以及一个可学习的标量 `x_logvar`。

和 diffusion transformer 的实现方式类似，我们会把编码器（以及解码器）设计成一系列 **block** 的组合。每个 block 都包含若干残差连接，接着是一个注意力层，再接一个下采样（或上采样）层，最后一个 block 除外。下面我们会按照“组件一个个来”的方式，先给出一段通俗描述，再请你补全实现细节。


### 问题 4.1：残差块
**你的任务：** 实现 **ResidualBlock** 类，使其满足下面这个步骤配方：

0. 先把输入保存为 `x_skip`，以便最后加回来。
1. 做某种层归一化。我们推荐 `nn.GroupNorm(...)`。当 `num_groups = 1` 时，它在效果上相当于对展平后的图像做 `nn.LayerNorm`。
2. 应用一个卷积核大小为 3 的卷积层。
3. 应用你喜欢的某种非线性激活函数。
4. 再应用一个最终的 1x1 卷积（卷积核大小为 1）。
5. 最后，返回上一步的输出与 `x_skip` 相加后的结果，以实现残差连接。


In [ ]:
class ResidualBlock(nn.Module):
  """ Two applications of LN + convolution + non-linearity + residual connection """
  def __init__(self, channels: int, act: nn.Module = nn.SiLU):
    super().__init__()

    # Init norm, convolutions, and activations
    raise NotImplementedError("Fill me in!")

    # Initialize the second convolution to zero - stabilizes training early on!
    nn.init.zeros_(self.conv2.weight)
    nn.init.zeros_(self.conv2.bias)

  def forward(self, x: torch.Tensor):
    # Res init
    x_skip = x

    # Norm
    raise NotImplementedError("Fill me in!")

    # First convolution
    raise NotImplementedError("Fill me in!")

    # Activation
    raise NotImplementedError("Fill me in!")

    # Second convolution
    raise NotImplementedError("Fill me in!")

    # Return residual connection
    return x_skip + x

### 问题 4.2：注意力块
**你的任务：** 实现 **AttnBlock** 类，使其满足下面这个步骤配方：

0. 把张量从图像形状 `b c h w` 重排为 token 列表形状 `b (h w) c`。这里你可能会觉得 `einops.rearrange` 很好用。
2. 把当前输入保存为 `x_skip`，供下一次残差连接使用。
3. 执行归一化和多头注意力（使用已有的 `MHA` 类）。
4. 利用前面保存的 `x_skip` 做残差连接。
5. 再次把当前输入保存为 `x_skip`，供下一次残差连接使用。
6. 执行归一化和前馈层。
7. 利用保存的 `x_skip` 做残差连接。


In [ ]:
class AttnBlock(nn.Module):
  def __init__(self, channels: int):
    super().__init__()

    # Reshape
    self.reshape1 = Rearrange('b c h w -> b (h w) c')

    # Norm + attention
    raise NotImplementedError("Fill me in!")


    # Norm + ff
    raise NotImplementedError("Fill me in!")


  def forward(self, x: torch.Tensor):
    b, c, h, w = x.shape
    x = self.reshape1(x)

    # Attention + residual connection
    raise NotImplementedError("Fill me in!")

    # Feedforward + residual connection
    raise NotImplementedError("Fill me in!")

    return rearrange(x, 'b (h w) c -> b c h w', h=h, w=w)

### 问题 4.3：编码器块
**你的任务：** 实现 **EncoderBlock** 类，使其满足下面这个步骤配方：

0. 应用两个残差块。
1. 应用一个注意力块。
2. 如果 `downsample_channels` 不是 `None`，就再应用一个下采样卷积（使用标准的 `nn.Conv2d`，并设定 `kernel_size=3`、`padding=1`、`stride=2` 就足够了）。


In [ ]:
class EncoderBlock(nn.Module):
  def __init__(self, in_channels: int, downsample_channels: Optional[int] = None):
    super().__init__()
    raise NotImplementedError("Fill me in!")

  def forward(self, x: torch.Tensor):
    raise NotImplementedError("Fill me in!")

### 问题 4.4：编码器
**你的任务：** 实现 **Encoder** 类，使其满足下面这个步骤配方：

0. 先做一个初始卷积，把输入通道数（MNIST 中为 1）映射到初始隐藏通道数。
1. 对每个隐藏通道依次应用一个 encoder block，除了最后一个 block 之外，其余都要做下采样。
2. 最后通过一个归一化层和一个输出 1x1 卷积层，预测 `z_mean`。
3. 令 `self.logvar = nn.Parameter(torch.zeros(()))`，并返回 `z_mean` 和 `self.logvar`。


In [ ]:
class Encoder(nn.Module):
  def __init__(self, in_channels: int, hidden_channels: list[int]):
    super().__init__()

    # Initial conv2d
    self.init_conv = nn.Conv2d(in_channels = in_channels, out_channels = hidden_channels[0], kernel_size=3, padding=1, stride=1)

    # Initialize channels
    ch_in = hidden_channels
    ch_out = hidden_channels[1:] + [None]
    blocks = []
    for in_c, out_c in zip(ch_in, ch_out):
      blocks.append(EncoderBlock(in_c, out_c))
    self.blocks = nn.ModuleList(blocks)

    # Predict z_mean
    z_dim = hidden_channels[-1]
    self.z_mean = nn.Sequential(
      nn.GroupNorm(1, z_dim),
      nn.Conv2d(in_channels = z_dim, out_channels = z_dim, kernel_size=1, stride=1, padding=0),
    )

    # Scalar log-variance
    self.logvar = nn.Parameter(torch.zeros(()))

  def forward(self, x: torch.Tensor):
    raise NotImplementedError("Fill me in!")


### 问题 4.5：解码器块
**你的任务：** 实现 **DecoderBlock** 类，使其满足下面这个步骤配方：

0. 应用两个残差块。
1. 应用一个注意力块。
2. 如果 `upsample_channels` 不是 `None`，就应用一个上采样模块，例如 `nn.Upsample` 后接一个标准卷积。


In [ ]:
class DecoderBlock(nn.Module):
  def __init__(self, in_channels: int, upsample_channels: Optional[int] = None):
    super().__init__()
    self.res1 = ResidualBlock(in_channels)
    self.res2 = ResidualBlock(in_channels)
    self.attn = AttnBlock(in_channels)
    if upsample_channels is not None:
      self.upsample = nn.Sequential(
        nn.Upsample(scale_factor=2, mode='nearest'),
        nn.Conv2d(in_channels=in_channels, out_channels=upsample_channels, kernel_size=3, padding=1, stride=1),
      )
    else:
      self.upsample = None

  def forward(self, x: torch.Tensor):
    raise NotImplementedError("Fill me in!")

### 问题 4.6：解码器
**你的任务：** 实现 **Decoder** 类，使其满足下面这个步骤配方：

0. 对每个隐藏通道依次应用一个 decoder block，除了最后一个 block 之外，其余都要做上采样。
1. 最后通过一个归一化层和一个输出 1x1 卷积层，预测 `x_mean`。
2. 令 `self.logvar = nn.Parameter(torch.zeros(()))`，并返回 `x_mean` 和 `self.logvar`。


In [ ]:
class Decoder(nn.Module):
  def __init__(self, out_channels: int, hidden_channels: list[int]):
    super().__init__()

    # Initialize channels
    ch_in = hidden_channels
    ch_out = hidden_channels[1:] + [None]
    blocks = []
    for in_c, out_c in zip(ch_in, ch_out):
      blocks.append(DecoderBlock(in_c, out_c))
    self.blocks = nn.ModuleList(blocks)

    # Predict mean
    x_dim = hidden_channels[-1]
    self.x_mean = nn.Sequential(
      nn.GroupNorm(1, x_dim),
      nn.Conv2d(in_channels = x_dim, out_channels = out_channels, kernel_size=1, stride=1, padding=0),
    )

    # Scalar log-variance
    self.logvar = nn.Parameter(torch.zeros(()))

  def forward(self, x: torch.Tensor):
    raise NotImplementedError("Fill me in!")

### 问题 4.7：拼装起来
最后，我们把上面的子组件组装起来，完成 `VAE` 类。

**你的任务：** 实现 `VAE.compute_loss(...)`，从而实现正文中给出的 $L_{\text{VAE}}(\phi, \theta)$（公式 display (85)）。


In [ ]:
class VAE(nn.Module):
  def __init__(self, data_channels: int, hidden_channels: list[int], beta: float = 0.1):
    super().__init__()
    self.beta = beta

    # Encoder
    self._encoder = Encoder(data_channels, hidden_channels)

    # Decoder
    self._decoder = Decoder(data_channels, list(reversed(hidden_channels)))

  def encode(self, x: torch.Tensor):
    return self._encoder(x)

  def decode(self, z: torch.Tensor):
    return self._decoder(z)

  def forward(self, x: torch.Tensor):
    z_mean, z_logvar = self.encode(x)
    z = z_mean + torch.exp(0.5 * z_logvar) * torch.randn_like(z_mean)
    x_mean, x_logvar = self.decode(z)
    return z_mean, z_logvar, x_mean, x_logvar

  def compute_loss(self, z_mean: torch.Tensor, z_logvar: torch.Tensor, x_mean: torch.Tensor, x_logvar: torch.Tensor, x_true: torch.Tensor):
    """ See display 85 from the text """
    # KL loss
    raise NotImplementedError("Fill me in!")

    # Reconstruction loss
    raise NotImplementedError("Fill me in!")

    return kl_loss + recon_loss

在训练之前，我们先创建一个新的 `Trainer` 子类，用它来训练 VAE，并写一个辅助函数，用于可视化学到的潜空间中的插值效果。


In [ ]:
class MNISTVAETrainer(Trainer):
  def __init__(self, mnist_sampleable: LabeledSampleable, batch_size: int = 64, **kwargs):
    super().__init__(**kwargs)
    self.mnist = mnist_sampleable
    self.batch_size = batch_size

  def get_train_loss(self):
    """ See display 85 from the text """
    x, y = self.mnist.sample(self.batch_size)
    z_mean, z_std, x_mean, x_std = self.model(x)
    return self.model.compute_loss(z_mean, z_std, x_mean, x_std, x)

  @torch.no_grad()
  def checkpoint(self, step: int):
    # Save model
    torch.save(self.model.state_dict(), os.path.join(self.output_dir, f'step_{step:06d}_model.pt'))
    torch.save(self.opt.state_dict(), os.path.join(self.output_dir, f'step_{step:06d}_opt.pt'))

    # Save output visualization, using x_mean as reconstruction
    b = 10
    x, _ = self.mnist.sample(b)
    _, _, x_mean, _ = self.model(x)
    x_all = torch.cat([x, x_mean], dim=0)
    grid = make_grid(x_all, nrow=b, normalize=True, value_range=(0,1))
    plt.imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    plt.axis("off")
    plt.title("VAE Reconstruction")
    plt.savefig(os.path.join(self.output_dir, f'step_{step:06d}_output.png'))
    plt.close()

@torch.no_grad()
def visualize_latent_interpolation(x1: torch.Tensor, x2: torch.Tensor, vae: VAE, n_steps: int, save_path: Optional[str] = None):
   z1_mean, z1_logvar = vae.encode(x1)
   z1 = z1_mean + torch.exp(0.5 * z1_logvar) * torch.randn_like(z1_mean) # 1 c h w

   z2_mean, z2_logvar = vae.encode(x2)
   z2 = z2_mean + torch.exp(0.5 * z2_logvar) * torch.randn_like(z2_mean) # 1 c h w

   lambdas = torch.linspace(0, 1, n_steps).to(z1.device)
   zs = (1 - lambdas) * z1.unsqueeze(-1) + lambdas * z2.unsqueeze(-1) # 1 c h w n_steps
   zs = rearrange(zs, '1 c h w n -> n c h w')
   samples, _ = vae.decode(zs) # n_steps 1 h w

   grid = make_grid(samples, nrow=n_steps, normalize=True, value_range=(0, 1))
   plt.figure(figsize=(n_steps * 2, 2))
   plt.imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
   plt.axis("off")
   plt.title("Latent Interpolation")
   if save_path is not None:
       plt.savefig(save_path)
       plt.close()
   else:
       plt.show()
   return samples

最后，让我们来训练这个 VAE。


In [ ]:
# Create dataset + VAE
device = torch.device('cuda')
mnist = MNISTSampler().to(device)
vae = VAE(
   data_channels = 1,
   hidden_channels = [16, 32, 64, 128],
   beta = 10.0,
).to(device)

# Create Trainer + train
trainer = MNISTVAETrainer(
    mnist_sampleable = mnist,
    batch_size = 64,
)
losses, steps = trainer.train(
   model = vae,
   num_steps = 5000,
   lr = 1e-3,
   warmup_steps = 500,
   ckpt_every = 250,
)
plt.plot(steps, losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.show()

In [ ]:
# Perform interpolation in the latent space
vae.eval()
samples, _ = mnist.sample(2)
interpolated_samples = visualize_latent_interpolation(
   x1 = samples[:1],
   x2 = samples[1:2],
   vae = vae,
   n_steps = 10,
) # n_steps 1 h w

# 第 5 部分：训练一个潜空间扩散模型
在这一节中，我们会在一个训练好的 VAE 的潜空间中训练 diffusion transformer。幸运的是，大部分工作前面都已经完成了！现在剩下的只是把所有部件拼起来，也就是把 VAE、潜空间概率路径和 diffusion transformer 组合到一起。


### 问题 5.1：潜空间扩散训练器
下面给出了 `LatentCFGTrainer` 的部分实现，它是 `CFGTrainer` 的一个扩展版本，用于在 VAE 的潜空间中训练。

**你的任务：** 实现 `LatentCFGTrainer.get_train_loss`。

**提示：**
1. 可以改写 `CFGTrainer.get_train_loss`：不同的是，这次不要调用 `self.path.p_data.sample(...)`，而是需要直接从 MNIST 中采样，然后把样本送入编码器。
2. 记得把对 `vae.encode` 的调用放在 `torch.no_grad()` 保护之下！


In [ ]:
class LatentCFGTrainer(Trainer):
    def __init__(
        self,
        mnist: MNISTSampler,
        vae: VAE,
        path: GaussianConditionalProbabilityPath,
        eta: float,
        null_label: int,
        eps: float = 0.001,
        **kwargs
    ):
        assert eta > 0 and eta < 1
        super().__init__(**kwargs)
        self.mnist = mnist
        self.vae = vae
        self.path = path
        self.eta = eta
        self.eps = eps
        self.path = path
        self.null_label = null_label

    @torch.no_grad()
    def visualize_samples(self, save_path: str, samples_per_class: int = 10, num_timesteps: int = 100, guidance_scales: List[float] = [1.0, 3.0, 5.0], use_tqdm = False):
      # Graph
      fig, axes = plt.subplots(1, len(guidance_scales), figsize=(10 * len(guidance_scales), 10))

      for idx, w in enumerate(guidance_scales):
          # Setup ode and simulator
          ode = CFGVectorFieldODE(self.model, guidance_scale=w, null_label=10)
          simulator = EulerSimulator(ode)

          # Sample initial conditions
          y = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=torch.int64).repeat_interleave(samples_per_class).to(device)
          num_samples = y.shape[0]
          z0 = self.path.p_simple.sample(num_samples)

          # Simulate
          ts = torch.linspace(0,0.999,num_timesteps).view(1, -1, 1, 1, 1).expand(num_samples, -1, 1, 1, 1).to(device)
          z1 = simulator.simulate(z0, ts, y=y, use_tqdm=use_tqdm)

          # Decode
          x1, _ = self.vae.decode(z1)

          # Plot
          v_min, v_max = x1.min(), x1.max()
          x1 = (x1 - v_min) / (v_max - v_min)
          grid = make_grid(x1, nrow=samples_per_class, normalize=True, value_range=(0,1))
          axes[idx].imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
          axes[idx].axis("off")
          axes[idx].set_title(f"Guidance: $w={w:.1f}$", fontsize=25)

      # Save
      if save_path is not None:
          plt.savefig(save_path)
          plt.close()
      else:
        plt.show()

    plt.show()

    def checkpoint(self, step: int):
      # Save model
      torch.save(self.model.state_dict(), os.path.join(self.output_dir, f'step_{step:6d}_model.pt'))
      torch.save(self.opt.state_dict(), os.path.join(self.output_dir, f'step_{step:6d}_opt.pt'))

      # Save output visualization
      self.visualize_samples(save_path=os.path.join(self.output_dir, f'step_{step:6d}_output.png'))

    def get_train_loss(self, batch_size: int) -> torch.Tensor:
        # Step 1: Sample z, y from MNIST + encode
        with torch.no_grad():
          raise NotImplementedError("Fill me in!")


        # Step 2: Set each label to 10 (i.e., null) with probability eta
        raise NotImplementedError("Fill me in!")


        # Step 3: Sample t and x
        raise NotImplementedError("Fill me in!")


        # Step 4: Regress and output loss
        raise NotImplementedError("Fill me in!")


In [ ]:
# Finally, let's train!

vae = vae.to(device) # VAE(data_channels = 1, hidden_channels = [16, 32, 64, 128], beta = 1.0)

# Initialize latent probability path
c = 128
img_size = 4

path = GaussianConditionalProbabilityPath(
    p_data = None,
    p_simple_shape = [c, img_size, img_size],
    alpha = LinearAlpha(),
    beta = LinearBeta()
).to(device)

# Initialize model
dit = DiffusionTransformerFlowModel(
    img_size = img_size,
    patch_size = 1,
    num_layers = 8,
    c = c,
    dim = 256,
    heads = 8,
    final_dim = 10,
    n_classes = 11,
).to(device)

# Dataset
mnist = MNISTSampler().to(device)

# Initialize trainer
trainer = LatentCFGTrainer(
    mnist = mnist,
    vae = vae,
    path = path,
    eta=0.35,
    null_label=10
)

# Train! You should have reasonable results in ~15 A100 minutes
losses, steps = trainer.train(model=dit, num_steps = 10000, lr=0.4e-3, batch_size=256, ckpt_every=500)

plt.plot(steps, losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Loss vs. Step")
plt.show()

### 参考文献：
1. William Peebles 和 Saining Xie. *Scalable Diffusion Models with Transformers*. 2023. arXiv: 2212.09748.
[cs.CV]. 链接：https://arxiv.org/abs/2212.09748.
2. Robin Rombach 等. *High-Resolution Image Synthesis with Latent Diffusion Models*. 2022. arXiv: 2112.10752.
[cs.CV]. 链接：https://arxiv.org/abs/2112.10752.
